# 🍶 Flask/Django — Web Development
## Python Ecosystem Tutorial Series — Module 17 of 18

**Author:** Himanshu Goel | [himanshugoel.github.io](https://himanshugoel.github.io)

---

| | |
|---|---|
| **Library** | 🍶 Flask/Django |
| **Domain** | Web Development |
| **Dataset** | Chemical API |
| **Module** | 17 of 18 |

**What you will learn:**

1. What Flask/Django is and why it exists
2. Core concepts and data structures
3. Hands-on code with real data
4. Visualisations and interpretation
5. When to use it and alternatives

```bash
# Install required libraries
pip install flask django
```

## Quick Reference Card

| Code | What it does |
|------|--------------|
| `@app.route("/path")` | Flask route |
| `request.get_json()` | Parse JSON body |
| `jsonify(data)` | Return JSON response |
| `Model.objects.filter()` | Django ORM query |
| `@admin.register()` | Django admin |

# 17. 🍶 Flask — Lightweight Web Development
> **Python + Flask = Lightweight Web Dev**

Flask is the micro web framework. Minimal, flexible, perfect for small APIs and web apps.
Django (its sibling) is the full-stack framework — includes ORM, admin panel, auth.

**Flask key concepts:** routes, templates, request, redirect, Jinja2  
**Django key concepts:** models, views, URLs, templates, admin, migrations

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# ── Flask app ─────────────────────────────────────────────────────────────────
flask_code = """
from flask import Flask, request, jsonify, render_template
import pandas as pd

app = Flask(__name__)  # That's it — Flask is running!

# ── Route 1: Home page ────────────────────────────────────────────────────────
@app.route("/")
def home():
    return render_template("index.html", title="Chemical Database")

# ── Route 2: GET single compound ─────────────────────────────────────────────
@app.route("/api/compound/<name>", methods=["GET"])
def get_compound(name):
    data = db.get(name.lower())
    if not data:
        return jsonify({"error": "Not found"}), 404
    return jsonify(data)

# ── Route 3: POST search ─────────────────────────────────────────────────────
@app.route("/api/search", methods=["POST"])
def search():
    body   = request.get_json()
    query  = body.get("query", "").lower()
    max_mw = body.get("max_mw", 500)
    results = [v for k,v in db.items()
               if query in k and v["mw"] <= max_mw]
    return jsonify({"results": results, "count": len(results)})

if __name__ == "__main__":
    app.run(debug=True, port=5000)
"""

# ── Django model ──────────────────────────────────────────────────────────────
django_code = """
# models.py — defines database tables as Python classes
from django.db import models

class Chemical(models.Model):
    name        = models.CharField(max_length=200)
    smiles      = models.TextField()
    mw          = models.FloatField()
    logp        = models.FloatField()
    category    = models.CharField(max_length=100)
    created_at  = models.DateTimeField(auto_now_add=True)

    class Meta:
        ordering = ["name"]

    def __str__(self):
        return self.name

# views.py — handles HTTP requests
from django.views.generic import ListView

class ChemicalListView(ListView):
    model    = Chemical
    template = "chemicals/list.html"

    def get_queryset(self):
        # Filter by URL query param: /chemicals/?category=drug
        category = self.request.GET.get("category")
        qs = super().get_queryset()
        if category:
            qs = qs.filter(category=category)
        return qs
"""

print("Flask code:")
print(flask_code)
print("\nDjango model + view:")
print(django_code)

# ── Flask vs Django comparison ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

features = ["Setup time","Flexibility","Built-in features","Learning curve","Performance","Community"]
flask_scores   = [9, 9, 3, 8, 7, 8]
django_scores  = [4, 5, 10, 4, 6, 10]

x = np.arange(len(features))
w = 0.35
axes[0].bar(x-w/2, flask_scores,  w, label="Flask",  color="#E74C3C", alpha=0.85, edgecolor="white")
axes[0].bar(x+w/2, django_scores, w, label="Django", color="#27AE60", alpha=0.85, edgecolor="white")
axes[0].set_xticks(x); axes[0].set_xticklabels(features, rotation=30, ha="right", fontsize=8)
axes[0].set_ylabel("Score (0-10)")
axes[0].set_title("Flask vs Django\nWhen to choose each", fontweight="bold")
axes[0].legend(); axes[0].grid(True, alpha=0.3, axis="y")
axes[0].axhline(7, c="grey", ls=":", alpha=0.5)

# Usage breakdown pie
uses = ["REST APIs\n(Flask)", "Full Apps\n(Django)", "Microservices\n(Flask)",
         "Admin panels\n(Django)", "Prototypes\n(Flask)"]
vals = [30, 25, 20, 15, 10]
cols = ["#E74C3C","#27AE60","#E74C3C","#27AE60","#E74C3C"]
wedges, texts, autotexts = axes[1].pie(vals, labels=uses, autopct="%1.0f%%",
                                        colors=cols, startangle=90, textprops={"fontsize":8})
axes[1].set_title("Typical Use Cases\n(Red=Flask, Green=Django)", fontweight="bold")

plt.suptitle("Flask + Django — Web Development Frameworks", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("flask_django.png", dpi=120, bbox_inches="tight")
plt.show()

## Deep Dive: Flask vs Django

### Architecture Philosophy
Flask is a microframework: it gives you request routing, response handling, and Jinja2 templates. Everything else (database, authentication, forms, email) you choose yourself. This flexibility is its strength and its burden.

Django is a "batteries included" framework: ORM, admin interface, user authentication, forms, email, migrations, caching, and i18n are all built in and integrated. You trade flexibility for speed of development.

### Flask Request Lifecycle
```python
1. Client sends HTTP request to /drugs/aspirin
2. Flask matches URL pattern to @app.route("/drugs/<name>")
3. Flask calls get_drug("aspirin")
4. Function returns jsonify(data)
5. Flask wraps in HTTP response with headers
6. Response sent back to client
```

### Django ORM Deep Dive
```python
# Filtering (all lazy — returns QuerySet, not data)
Chemical.objects.filter(category="drug")      # WHERE category='drug'
Chemical.objects.exclude(status="banned")     # WHERE NOT status='banned'
Chemical.objects.filter(mw__lte=500)          # WHERE mw <= 500
Chemical.objects.filter(name__icontains="ol") # WHERE LOWER(name) LIKE '%ol%'

# Chaining
Chemical.objects.filter(category="drug").exclude(status="banned").order_by("mw")[:10]
```

### Django Admin Free Features
With 3 lines of code you get: list view with search and filters, create/edit/delete forms with validation, bulk action support, history/audit log of all changes.

### Decision Heuristic
If you need a REST API only: FastAPI (best) or Flask. If you need a website with user accounts, admin interface, and a database: Django.


## ✅ Key Takeaways — 🍶 Flask/Django

1. Flask gives maximum flexibility; Django gives maximum speed of development
2. Django ORM is safe by default — all queries are parameterised (no SQL injection)
3. Django Admin is a free CRUD interface — 3 lines of code for a full database manager
4. FastAPI is the modern choice for REST APIs; use Flask/Django for full web apps

---
*Next: Continue to Module 18 of 18 in the Python Ecosystem Tutorial Series*  
*Portfolio: [himanshugoel.github.io](https://himanshugoel.github.io)*